# OCTO 

In [ ]:
import pymaid

# Load the pymaid configuration

volumes = [
    {
        "name": "MR143",
        "url": "https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/",
        "project": "19",
        "token": "x"
    },
    {
        "name": "Octo",
        "url": "https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/",
        "project": "18",
        "token": "x"
    }
]
http_user = "x"
http_pass = "x"


In [187]:

rm_octo = pymaid.CatmaidInstance(volumes[1]["url"], project_id=volumes[1]["project"], api_token=volumes[1]["token"],
                                 http_user=http_user, http_password=http_pass)
rm_octo


INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


CatmaidInstance at 6488490384.
Server: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem
Project: 18
Caching True (size limit 128; time limit None)
Cache size: 0.0

In [188]:
annotations_list_octo = ["annotation:cube3: pushed fp synapses", "annotation:cube2: pushed fp synapses",
                         "annotation:cube1: pushed fp synapses"]


In [189]:
import pymaid
import requests
import urllib

import pandas as pd
def get_transactions(range_start=None, range_length=25, remote_instance=None):
    """
    Retrieve transactions from Catmaid. Rewrite pymaid.get_transactions to include millisecond time."""
    remote_instance = pymaid.utils._eval_remote_instance(remote_instance)
    remote_transactions_url = remote_instance._get_transactions_url()

    desc = {'range_start': range_start, 'range_length': range_length}
    desc = {k: v for k, v in desc.items() if v is not None}
    remote_transactions_url += '?%s' % urllib.parse.urlencode(desc)

    data = remote_instance.fetch(remote_transactions_url)
    df = pd.DataFrame.from_dict(data['transactions'])

    user_list = pymaid.get_user_list(remote_instance=remote_instance)
    user_dict = user_list.set_index('id').login.to_dict()
    df['user'] = df.user_id.map(user_dict)

    # Preserve milliseconds and timezone
    df['execution_time'] = pd.to_datetime(df['execution_time'], format='%Y-%m-%dT%H:%M:%S.%f%z')

    return df


def _get_transactions_location_url(self, **GET):
    """Generate url to get transactions (GET)."""
    return self.make_url(self.project_id, 'transactions/location', **GET)

# Add the method to the CatmaidInstance class
setattr(pymaid.CatmaidInstance, '_get_transactions_location_url', _get_transactions_location_url)


def get_transactions_with_locations(remote_instance, project_id, range_length=500):
    """
    Retrieve all transactions (using pymaid.get_transactions) and add location details
    for each transaction.

    Args:
        remote_instance: The remote instance configuration for pymaid.
        project_id (int): The Catmaid project ID.
        range_length (int): How many transactions to retrieve.

    Returns:
        list: List of transaction dictionaries, each enriched with location data.
    """
    # Fetch transactions from pymaid.
    # transactions = pymaid.get_transactions(remote_instance=remote_instance, range_length=range_length)
    transactions = get_transactions(remote_instance=remote_instance, range_length=range_length)
    # print(type(transactions))

    enriched_transactions = []
    for i, transaction in transactions.iterrows(): # transaction is a df
        # print(transaction)
        transaction_id = transaction.transaction_id
        execution_time = transaction.execution_time
        
        # print(transaction_id, execution_time)

        # Make sure the required keys are available.
        if not transaction_id or not execution_time:
            continue

        params = urllib.parse.urlencode({
        "transaction_id": transaction_id,
        "execution_time": execution_time
        })

        # Build URL for location details
        url = f"{remote_instance.server}/{remote_instance.project_id}/transactions/location?{params}"

        # If pymaid has a lower-level GET method, use it:
        try:
            # Use the built-in fetch method to retrieve data.
            location_data = remote_instance.fetch(url, return_type="json")

            # print(location_data)
            #pymaid._get(url, remote_instance=remote_instance)
        except AttributeError:
            # Otherwise, fall back to using requests directly (ensure proper authentication as needed)
            location_response = requests.get(url, headers=pymaid.get_headers(remote_instance))
            location_response.raise_for_status()
            location_data = location_response.json()
        except Exception as e:
            print(f"Error fetching location data for transaction {transaction_id}: {e}")
            location_data = {'x': None,'y': None, 'z': None}

        # Append the location details to the transaction.
        transaction["location_data"] = location_data
        enriched_transactions.append(transaction)
    
    return enriched_transactions


# ----- Usage Example -----

# Set up your remote instance and project id (replace with your actual settings)
rm_octo = pymaid.CatmaidInstance("https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/", project_id=18,
                                 api_token="4f640a9240861b412ebcab8cc01f574e5aee16c3",
                                 http_user="smohinta", http_password="headset-recovery-handshake")
project_id = 18

# Now call the function to retrieve enriched transactions.
try:
    txns_with_locations = get_transactions_with_locations(rm_octo, project_id, range_length=500)
    print(txns_with_locations)
    for txn in txns_with_locations:
        print("Transaction ID:", txn.get("transaction_id"))
        print("Location Data:", txn.get("location_data"))
        print("=" * 40)
except Exception as e:
    print("An error occurred:", e)


INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


Error fetching location data for transaction 2506894: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/18/transactions/location?transaction_id=2506894&execution_time=2025-03-07+14%3A32%3A10.942889%2B00%3A00
Error fetching location data for transaction 2506885: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/18/transactions/location?transaction_id=2506885&execution_time=2025-03-07+14%3A31%3A49.441345%2B00%3A00
Error fetching location data for transaction 2506881: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/18/transactions/location?transaction_id=2506881&execution_time=2025-03-07+14%3A31%3A23.746204%2B00%3A00
Error fetching location data for transaction 2506840: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/18/transactions/loca

KeyboardInterrupt: 

In [ ]:
import pandas as pd
output_dir = "/Users/sam/Library/CloudStorage/OneDrive-UniversityofCambridge/Synapse_localisation/synapse_curation/catmaid_tracker_plots"

# Convert to DataFrame
df = pd.DataFrame(txns_with_locations)

# Print the DataFrame
display(df)

# Save the DataFrame to a CSV file
df.to_csv(f"{output_dir}/transactions_with_locations.csv", index=False)

# group by user and label and count the number of transactions
grouped = df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped

,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data
0,2508005,2025-03-08 14:58:34.969793+00:00,32,18,Backend,treenodes.create,smohinta-su,"{'x': 48818.43, 'y': 26986.898, 'z': 59824.0}"
1,2507633,2025-03-07 17:15:50.066278+00:00,32,18,Backend,treenodes.remove,smohinta-su,"{'x': 49088.297, 'y': 27328.594, 'z': 59960.0}"
2,2506894,2025-03-07 14:32:10.942889+00:00,32,18,Backend,annotations.add,smohinta-su,"{'x': None, 'y': None, 'z': None}"
3,2506885,2025-03-07 14:31:49.441345+00:00,32,18,Backend,annotations.add,smohinta-su,"{'x': None, 'y': None, 'z': None}"
4,2506881,2025-03-07 14:31:23.746204+00:00,32,18,Backend,annotations.add,smohinta-su,"{'x': None, 'y': None, 'z': None}"
...,...,...,...,...,...,...,...,...
495,2488025,2025-02-26 13:51:36.073090+00:00,38,18,Backend,links.create,shiyan,"{'x': 66568.0, 'y': 48168.0, 'z': 39632.0}"
496,2488024,2025-02-26 13:51:36.066285+00:00,38,18,Backend,links.create,shiyan,"{'x': 66680.0, 'y': 48168.0, 'z': 39200.0}"
497,2488023,2025-02-26 13:51:36.060833+00:00,38,18,Backend,links.create,shiyan,"{'x': 66672.0, 'y': 47616.0, 'z': 39096.0}"
498,2488022,2025-02-26 13:51:36.008623+00:00,38,18,Backend,links.create,shiyan,"{'x': 66664.0, 'y': 52104.0, 'z': 40488.0}"


,user,label,counts
0,acardona,annotations.add,2
1,acardona,annotations.remove,2
2,acardona,treenodes.create,3
3,acardona,treenodes.remove,3
4,adulac,annotations.add,74
5,gmo,treenodes.create,5
6,gmo,treenodes.remove,5
7,hack_guest,annotations.add,30
8,hack_guest,treenodes.create,1
9,hack_guest,treenodes.remove,1


In [ ]:
# Filter transactions on the hackathon day
import pandas as pd
from datetime import datetime


# Convert execution_time to datetime if it's not already
df['execution_time'] = pd.to_datetime(df['execution_time'])

# Get today's date
today = pd.Timestamp(2025, 3, 6).date() #datetime.now().date()
print(today)

# Filter for today's transactions
today_df = df[df['execution_time'].dt.date == today]

# drop one user- smohinta-su
today_df = today_df[today_df['user'] != 'smohinta-su']
today_df = today_df[today_df['user'] != 'smohinta']


# replace hack_guest with sharris
today_df['user'] = today_df['user'].replace({'hack_guest': 'sharris'})

# replace realnames with anonymised reviewer names
today_df['user_anon'] = today_df['user'].replace({'mclayton': 'Reviewer 1', 'mrobbins': 'Reviewer 2', 'gmo': 'Reviewer 3', 'adulac': 'Reviewer 4',
                                             'nceffa': 'Reviewer 5',
                                             'sharris': 'Reviewer 6',
                                             'swilson': 'Reviewer 7',
                                             'shiyan': 'Reviewer 8',
                                             'hack_guest': 'Reviewer 9'
                                             }) 

display(today_df)


# group by user and label and count the number of transactions
grouped = today_df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped # this looks correct now based on our expectations from talking to the reviewers


2025-03-06


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon
7,2504056,2025-03-06 19:15:08.061896+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104664.0, 'y': 53920.0, 'z': 32096.0}",Reviewer 6
8,2504054,2025-03-06 19:10:09.244155+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102224.0, 'y': 52888.0, 'z': 32264.0}",Reviewer 6
9,2504052,2025-03-06 19:07:56.060628+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102064.0, 'y': 52872.0, 'z': 32280.0}",Reviewer 6
10,2504050,2025-03-06 19:05:27.749166+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104480.0, 'y': 52576.0, 'z': 32016.0}",Reviewer 6
11,2504048,2025-03-06 19:04:12.100705+00:00,47,18,Backend,annotations.add,sharris,"{'x': 101040.0, 'y': 52128.0, 'z': 32576.0}",Reviewer 6
...,...,...,...,...,...,...,...,...,...
238,2501817,2025-03-06 16:14:08.303119+00:00,26,18,Backend,treenodes.create,nceffa,"{'x': 49178.332, 'y': 26878.238, 'z': 59824.0}",Reviewer 5
241,2501787,2025-03-06 16:08:20.815817+00:00,44,18,Backend,treenodes.remove,swilson,"{'x': 48634.062, 'y': 26615.21, 'z': 59896.0}",Reviewer 7
242,2501786,2025-03-06 16:08:19.527975+00:00,44,18,Backend,treenodes.remove,swilson,"{'x': 48774.984, 'y': 26671.07, 'z': 59896.0}",Reviewer 7
243,2501785,2025-03-06 16:02:44.086140+00:00,44,18,Backend,treenodes.create,swilson,"{'x': 48774.984, 'y': 26671.07, 'z': 59896.0}",Reviewer 7


,user,label,counts
0,adulac,annotations.add,74
1,gmo,treenodes.create,5
2,gmo,treenodes.remove,5
3,mclayton,annotations.add,63
4,mclayton,links.create,4
5,mclayton,links.remove,1
6,mclayton,nodes.update_location,1
7,mclayton,treenodes.create,9
8,mclayton,treenodes.remove,2
9,mrobbins,annotations.add,24


In [ ]:
# Filter only annotations:add from transactions because we are interested in synapse annotations

today_df_filt = today_df[today_df['label'] == 'annotations.add']
display(today_df_filt)


# Again group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon
7,2504056,2025-03-06 19:15:08.061896+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104664.0, 'y': 53920.0, 'z': 32096.0}",Reviewer 6
8,2504054,2025-03-06 19:10:09.244155+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102224.0, 'y': 52888.0, 'z': 32264.0}",Reviewer 6
9,2504052,2025-03-06 19:07:56.060628+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102064.0, 'y': 52872.0, 'z': 32280.0}",Reviewer 6
10,2504050,2025-03-06 19:05:27.749166+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104480.0, 'y': 52576.0, 'z': 32016.0}",Reviewer 6
11,2504048,2025-03-06 19:04:12.100705+00:00,47,18,Backend,annotations.add,sharris,"{'x': 101040.0, 'y': 52128.0, 'z': 32576.0}",Reviewer 6
...,...,...,...,...,...,...,...,...,...
211,2502288,2025-03-06 16:39:42.044296+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': None, 'y': None, 'z': None}",Reviewer 2
218,2502261,2025-03-06 16:38:18.985573+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 47384.0, 'y': 26144.0, 'z': 65248.0}",Reviewer 2
225,2502239,2025-03-06 16:37:48.335028+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': None, 'y': None, 'z': None}",Reviewer 2
228,2502229,2025-03-06 16:37:26.702852+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': None, 'y': None, 'z': None}",Reviewer 2


,user,label,counts
0,adulac,annotations.add,74
1,mclayton,annotations.add,63
2,mrobbins,annotations.add,24
3,sharris,annotations.add,30


In [ ]:
# Save the filtered DataFrame to a CSV file
today_df_filt.to_csv(f"{output_dir}/transactions_with_locations_filtered.csv", index=False)

In [ ]:
# Check how many location_data = {'x': None, 'y': None, 'z': None} are there
display(today_df_filt['location_data'].value_counts())

# Which users have location_data = {'x': None, 'y': None, 'z': None}
location_missing = today_df_filt[today_df_filt['location_data'] == {'x': None, 'y': None, 'z': None}]
# group by user and label and count the number of transactions
grouped_missing = location_missing.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_missing)


# Filter out all missing locations
today_df_filt = today_df_filt[today_df_filt['location_data'] != {'x': None, 'y': None, 'z': None}]
display(today_df_filt)

# Extract x, y, z coordinates into separate columns
today_df_filt['post_x'] = today_df_filt['location_data'].apply(lambda x: x['x'])
today_df_filt['post_y'] = today_df_filt['location_data'].apply(lambda x: x['y'])
today_df_filt['post_z'] = today_df_filt['location_data'].apply(lambda x: x['z'])
display(today_df_filt)


# Group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


location_data
{'x': None, 'y': None, 'z': None}             18
{'x': 68896.0, 'y': 51776.0, 'z': 37816.0}     3
{'x': 66088.0, 'y': 49392.0, 'z': 38336.0}     2
{'x': 65160.0, 'y': 49448.0, 'z': 38008.0}     2
{'x': 68456.0, 'y': 47440.0, 'z': 37912.0}     2
                                              ..
{'x': 65336.0, 'y': 50064.0, 'z': 38600.0}     1
{'x': 65840.0, 'y': 49704.0, 'z': 38600.0}     1
{'x': 67584.0, 'y': 47872.0, 'z': 41816.0}     1
{'x': 66336.0, 'y': 49640.0, 'z': 39216.0}     1
{'x': 45048.0, 'y': 30288.0, 'z': 65232.0}     1
Name: count, Length: 164, dtype: int64

,user,label,counts
0,adulac,annotations.add,1
1,mclayton,annotations.add,10
2,mrobbins,annotations.add,6
3,sharris,annotations.add,1


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon
7,2504056,2025-03-06 19:15:08.061896+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104664.0, 'y': 53920.0, 'z': 32096.0}",Reviewer 6
8,2504054,2025-03-06 19:10:09.244155+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102224.0, 'y': 52888.0, 'z': 32264.0}",Reviewer 6
9,2504052,2025-03-06 19:07:56.060628+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102064.0, 'y': 52872.0, 'z': 32280.0}",Reviewer 6
10,2504050,2025-03-06 19:05:27.749166+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104480.0, 'y': 52576.0, 'z': 32016.0}",Reviewer 6
11,2504048,2025-03-06 19:04:12.100705+00:00,47,18,Backend,annotations.add,sharris,"{'x': 101040.0, 'y': 52128.0, 'z': 32576.0}",Reviewer 6
...,...,...,...,...,...,...,...,...,...
204,2502353,2025-03-06 16:42:47.417239+00:00,18,18,Backend,annotations.add,adulac,"{'x': 68456.0, 'y': 47440.0, 'z': 37912.0}",Reviewer 4
209,2502318,2025-03-06 16:41:27.477313+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 45792.0, 'y': 30640.0, 'z': 65080.0}",Reviewer 2
210,2502305,2025-03-06 16:40:45.982455+00:00,23,18,Backend,annotations.add,mclayton,"{'x': 69088.0, 'y': 47448.0, 'z': 38048.0}",Reviewer 1
218,2502261,2025-03-06 16:38:18.985573+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 47384.0, 'y': 26144.0, 'z': 65248.0}",Reviewer 2


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon,post_x,post_y,post_z
7,2504056,2025-03-06 19:15:08.061896+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104664.0, 'y': 53920.0, 'z': 32096.0}",Reviewer 6,104664.0,53920.0,32096.0
8,2504054,2025-03-06 19:10:09.244155+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102224.0, 'y': 52888.0, 'z': 32264.0}",Reviewer 6,102224.0,52888.0,32264.0
9,2504052,2025-03-06 19:07:56.060628+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102064.0, 'y': 52872.0, 'z': 32280.0}",Reviewer 6,102064.0,52872.0,32280.0
10,2504050,2025-03-06 19:05:27.749166+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104480.0, 'y': 52576.0, 'z': 32016.0}",Reviewer 6,104480.0,52576.0,32016.0
11,2504048,2025-03-06 19:04:12.100705+00:00,47,18,Backend,annotations.add,sharris,"{'x': 101040.0, 'y': 52128.0, 'z': 32576.0}",Reviewer 6,101040.0,52128.0,32576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
204,2502353,2025-03-06 16:42:47.417239+00:00,18,18,Backend,annotations.add,adulac,"{'x': 68456.0, 'y': 47440.0, 'z': 37912.0}",Reviewer 4,68456.0,47440.0,37912.0
209,2502318,2025-03-06 16:41:27.477313+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 45792.0, 'y': 30640.0, 'z': 65080.0}",Reviewer 2,45792.0,30640.0,65080.0
210,2502305,2025-03-06 16:40:45.982455+00:00,23,18,Backend,annotations.add,mclayton,"{'x': 69088.0, 'y': 47448.0, 'z': 38048.0}",Reviewer 1,69088.0,47448.0,38048.0
218,2502261,2025-03-06 16:38:18.985573+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 47384.0, 'y': 26144.0, 'z': 65248.0}",Reviewer 2,47384.0,26144.0,65248.0


,user,label,counts
0,adulac,annotations.add,73
1,mclayton,annotations.add,53
2,mrobbins,annotations.add,18
3,sharris,annotations.add,29


In [ ]:
# Find all neurons_ids and locations based on annotations list
import pandas as pd
def get_neurons_from_annotations_list(annotations_list, remote_instance):
    """
    Get all neurons and their locations based on a list of annotations.
    """
    neurons = []
    for annotation in annotations_list:
        # Get all annotations with the given name
        annotation_neurons = pymaid.get_neurons(annotation, remote_instance=remote_instance)
        neurons.extend(annotation_neurons)
    return neurons

neurons = get_neurons_from_annotations_list(annotations_list_octo, rm_octo)
print(neurons)

# Assuming these are all post neurons 
post = [neuron.id for neuron in neurons]
for i, neuron in enumerate(neurons):
    node_coords = neuron.nodes[['x', 'y', 'z']]
    print(f"Coordinates for neuron {neuron.skeleton_id}:")
    print(node_coords)
    print("=" * 40)
    if i == 2:
        break


# Create a list to store all neuron data
neuron_data = []

for neuron in neurons:
    # Get coordinates for each node in the neuron
    node_coords = neuron.nodes[['x', 'y', 'z']]
    
    # For each node in the neuron, create a row with all information
    for idx, coords in node_coords.iterrows():
        neuron_data.append({
            'neuron_id': neuron.id,
            'skeleton_id': neuron.skeleton_id,
            'name': neuron.name,
            'x': coords['x'],
            'y': coords['y'],
            'z': coords['z'],
            'node_id': idx
        })

# Convert to DataFrame
post_neurons_all_from_ann_df = pd.DataFrame(neuron_data)
display(post_neurons_all_from_ann_df)

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


Fetch neurons:   0%|          | 0/214 [00:00<?, ?it/s]

Make nrn:   0%|          | 0/214 [00:00<?, ?it/s]

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


Fetch neurons:   0%|          | 0/286 [00:00<?, ?it/s]

...

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


Fetch neurons:   0%|          | 0/215 [00:00<?, ?it/s]

Make nrn:   0%|          | 0/215 [00:00<?, ?it/s]

[type            CatmaidNeuron
name            neuron 605697
id                     605696
n_nodes                     1
n_connectors                1
n_branches                  0
n_leafs                     0
cable_length              0.0
soma                     None
units             1 nanometer
dtype: object, type            CatmaidNeuron
name            neuron 606209
id                     606208
n_nodes                     1
n_connectors                1
n_branches                  0
n_leafs                     0
cable_length              0.0
soma                     None
units             1 nanometer
dtype: object, type            CatmaidNeuron
name            neuron 605701
id                     605700
n_nodes                     1
n_connectors                1
n_branches                  0
n_leafs                     0
cable_length              0.0
soma                     None
units             1 nanometer
dtype: object, type            CatmaidNeuron
name            neuron 6

,neuron_id,skeleton_id,name,x,y,z,node_id
0,605696,605696,neuron 605697,45120.0,30320.0,60424.0,0
1,606208,606208,neuron 606209,45240.0,30352.0,63824.0,0
2,605700,605700,neuron 605701,48608.0,30632.0,60336.0,0
3,606212,606212,neuron 606213,45072.0,30752.0,63744.0,0
4,605704,605704,neuron 605705,48088.0,30632.0,60368.0,0
...,...,...,...,...,...,...,...
711,612337,612337,neuron 612338,66944.0,51448.0,41872.0,0
712,611829,611829,neuron 611830,69568.0,47360.0,39240.0,0
713,612341,612341,neuron 612342,67088.0,51568.0,41888.0,0
714,611833,611833,neuron 611834,69528.0,47544.0,39408.0,0


In [ ]:
# Filter the post_neurons_all_from_ann_df based on the post neurons we have from today_df_filt based on locations
# Create a mask for matching coordinates
matching_coords = post_neurons_all_from_ann_df.apply(
    lambda row: any((row['x'] == today_df_filt['post_x']) & 
                   (row['y'] == today_df_filt['post_y']) & 
                   (row['z'] == today_df_filt['post_z'])), 
    axis=1
)

# Filter post_neurons_all_from_ann_df to keep only the matching rows
post_neurons_all_from_ann_df_filterby_today_df = post_neurons_all_from_ann_df[matching_coords]
display(post_neurons_all_from_ann_df_filterby_today_df)


,neuron_id,skeleton_id,name,x,y,z,node_id
71,606348,606348,neuron 606349,45688.0,30680.0,64360.0,0
73,606352,606352,neuron 606353,45792.0,30744.0,64320.0,0
79,606364,606364,neuron 606365,45936.0,30376.0,64648.0,0
81,606368,606368,neuron 606369,45968.0,30448.0,64656.0,0
83,606372,606372,neuron 606373,47320.0,26240.0,65176.0,0
...,...,...,...,...,...,...,...
705,612325,612325,neuron 612326,69392.0,51488.0,41824.0,0
707,612329,612329,neuron 612330,69800.0,51720.0,41944.0,0
709,612333,612333,neuron 612334,69552.0,51776.0,42048.0,0
711,612337,612337,neuron 612338,66944.0,51448.0,41872.0,0


In [ ]:
post = [neuron.skeleton_id for i, neuron in post_neurons_all_from_ann_df_filterby_today_df.iterrows()]

# Get all connections for the neurons
connections = pymaid.get_connectors(post, remote_instance=rm_octo)
display(connections)

connector_ids = connections['connector_id'].tolist()
connector_details = pymaid.get_connector_details(connector_ids)
display(connector_details)

# Step 2: Rename `postsynaptic_to` to `skeleton_id` for clarity
connector_details = connector_details.rename(columns={'postsynaptic_to': 'skeleton_id'})

# Step 3: Convert skeleton_id to numeric type. Note this cannot work when > 1 skeleton_id is in list
# Extract the first (and presumably only) element from each list in the 'postsynaptic_to' column
connector_details['skeleton_id'] = connector_details['skeleton_id'].apply(lambda x: x[0] if x else None)

connector_details['skeleton_id'] = connector_details['skeleton_id'].astype(int)

# Step 4: Merge with connectors DataFrame (if needed)
connectors_with_skid = pd.merge(connections, connector_details[['connector_id', 'skeleton_id']], 
                                on='connector_id', how='left')
display(connectors_with_skid)

print(f"dtype connectors_with_skid['skeleton_id'].dtype: {connectors_with_skid['skeleton_id'].dtype}")
print(f"post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype: {post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype}")

# Convert to numeric type
post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'] = post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].astype(int)

# Step 5: Merge with post_neurons_df. Note connector = pre
final_post_conn_df_after_trans_filt = pd.merge(connectors_with_skid, post_neurons_all_from_ann_df_filterby_today_df, 
                  on='skeleton_id', how='left', suffixes=('_connector', '_post'))

display(final_post_conn_df_after_trans_filt)


final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.rename(columns={
    'x_connector': 'connector_x',
    'y_connector': 'connector_y',
    'z_connector': 'connector_z',
    'x_post': 'post_x',
    'y_post': 'post_y',
    'z_post': 'post_z'
})
print("result")
display(final_post_conn_df_after_trans_filt)


INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


,connector_id,x,y,z,confidence,creation_time,edition_time,tags,type,creator,editor
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,shiyan
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,shiyan
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,shiyan
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,shiyan
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,shiyan
...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,shiyan
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,shiyan
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,shiyan
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,shiyan


CN details:   0%|          | 0/162 [00:00<?, ?it/s]

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Data for 162 of 162 unique connector IDs retrieved (pymaid)


,connector_id,presynaptic_to,postsynaptic_to,presynaptic_to_node,postsynaptic_to_node
0,2109930,None,[606348],None,[2118035]
1,2109931,None,[606352],None,[2118036]
2,2109934,None,[606364],None,[2118039]
3,2109935,None,[606368],None,[2118040]
4,2109936,None,[606372],None,[2118041]
...,...,...,...,...,...
157,2119721,None,[611549],None,[2119504]
158,2119773,None,[611757],None,[2119556]
159,2119774,None,[611761],None,[2119557]
160,2119775,None,[611765],None,[2119558]


,connector_id,x,y,z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,shiyan,606348
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,shiyan,606352
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,shiyan,606364
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,shiyan,606368
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,shiyan,606372
...,...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,shiyan,612349
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,shiyan,612353
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,shiyan,612357
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,shiyan,612361


dtype connectors_with_skid['skeleton_id'].dtype: int64
post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype: int64


,connector_id,x_connector,y_connector,z_connector,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,x_post,y_post,z_post,node_id
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,shiyan,606348,606348,neuron 606349,45688.0,30680.0,64360.0,0
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,shiyan,606352,606352,neuron 606353,45792.0,30744.0,64320.0,0
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,shiyan,606364,606364,neuron 606365,45936.0,30376.0,64648.0,0
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,shiyan,606368,606368,neuron 606369,45968.0,30448.0,64656.0,0
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,shiyan,606372,606372,neuron 606373,47320.0,26240.0,65176.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,shiyan,612349,612349,neuron 612350,65600.0,51520.0,41920.0,0
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,shiyan,612353,612353,neuron 612354,65592.0,51400.0,41952.0,0
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,shiyan,612357,612357,neuron 612358,65744.0,51336.0,41928.0,0
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,shiyan,612361,612361,neuron 612362,67544.0,52096.0,41856.0,0


result


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,shiyan,606348,606348,neuron 606349,45688.0,30680.0,64360.0,0
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,shiyan,606352,606352,neuron 606353,45792.0,30744.0,64320.0,0
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,shiyan,606364,606364,neuron 606365,45936.0,30376.0,64648.0,0
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,shiyan,606368,606368,neuron 606369,45968.0,30448.0,64656.0,0
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,shiyan,606372,606372,neuron 606373,47320.0,26240.0,65176.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,shiyan,612349,612349,neuron 612350,65600.0,51520.0,41920.0,0
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,shiyan,612353,612353,neuron 612354,65592.0,51400.0,41952.0,0
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,shiyan,612357,612357,neuron 612358,65744.0,51336.0,41928.0,0
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,shiyan,612361,612361,neuron 612362,67544.0,52096.0,41856.0,0


In [ ]:
# Assuming you have a DataFrame 'final_post_conn_df_after_trans_filt' with 'skeleton_id' columns
skeleton_ids = final_post_conn_df_after_trans_filt['skeleton_id'].tolist()

# Get annotations for skeletons
skeleton_annotations = pymaid.get_annotations(skeleton_ids)
print(skeleton_annotations)


# Assuming your annotations are in a dictionary called 'annotations'
def get_relevant_annotations(anno_list):
    return [a for a in anno_list if a not in ['cube2: pushed fp synapses', 'pushed false positives synapses', 
                                              'cube1 : pushed fp synapses', 'cube3: pushed fp synapses']]

# Convert annotations dictionary to DataFrame
anno_data = []
for skeleton_id, annotations in skeleton_annotations.items():
    anno_data.append({
        'skeleton_id': skeleton_id,
        'annotations': annotations
    })

anno_df = pd.DataFrame(anno_data)
display(anno_df)

# Convert skeleton_id to integer
anno_df['skeleton_id'] = anno_df['skeleton_id'].astype(int)

# Check number of rows before merge
print("Rows in final_post_conn_df_after_trans_filt before merge:", len(final_post_conn_df_after_trans_filt))
print("Rows in anno_df:", len(anno_df))

# Check for duplicate skeleton_ids in both DataFrames
print("\nDuplicate skeleton_ids in final_post_conn_df_after_trans_filt:")
print(final_post_conn_df_after_trans_filt['skeleton_id'].value_counts())
print("\nDuplicate skeleton_ids in anno_df:")
print(anno_df['skeleton_id'].value_counts())


# Add annotations to the `final_post_conn_df_after_trans_filt` DataFrame
final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.merge(anno_df, on='skeleton_id', how='inner')

display(final_post_conn_df_after_trans_filt)

{'608126': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'distanced set', 'sh: distanced set'], '608131': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'pre correct', 'sh: distanced set'], '608135': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'wrong set', 'sh: wrong set'], '608139': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'distanced set', 'sh: uncertain'], '608143': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'wrong set', 'sh: wrong set'], '608147': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'distanced set', 'sh: distanced set'], '608151': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'distanced set', 'sh: wrong set'], '608155': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'pre correct', 'sh: pre correct'], '608159': ['cube2: pushed fp synapses', 'pushed false positives synapses', 'pre correct', 'sh: distanced set'], '6081

,skeleton_id,annotations
0,608126,"[cube2: pushed fp synapses, pushed false posit..."
1,608131,"[cube2: pushed fp synapses, pushed false posit..."
2,608135,"[cube2: pushed fp synapses, pushed false posit..."
3,608139,"[cube2: pushed fp synapses, pushed false posit..."
4,608143,"[cube2: pushed fp synapses, pushed false posit..."
...,...,...
157,611549,"[pushed false positives synapses, pre correct,..."
158,611757,"[pushed false positives synapses, cube1: pushe..."
159,611761,"[pushed false positives synapses, cube1: pushe..."
160,611765,"[pushed false positives synapses, cube1: pushe..."


Rows in final_post_conn_df_after_trans_filt before merge: 162
Rows in anno_df: 162

Duplicate skeleton_ids in final_post_conn_df_after_trans_filt:
skeleton_id
606348    1
611797    1
611725    1
611729    1
611733    1
         ..
611533    1
611537    1
611541    1
611545    1
612365    1
Name: count, Length: 162, dtype: int64

Duplicate skeleton_ids in anno_df:
skeleton_id
608126    1
612365    1
611585    1
611785    1
611625    1
         ..
612213    1
612333    1
611657    1
611653    1
612237    1
Name: count, Length: 162, dtype: int64


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id,annotations
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,shiyan,606348,606348,neuron 606349,45688.0,30680.0,64360.0,0,"[cube3: pushed fp synapses, pushed false posit..."
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,shiyan,606352,606352,neuron 606353,45792.0,30744.0,64320.0,0,"[cube3: pushed fp synapses, pushed false posit..."
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,shiyan,606364,606364,neuron 606365,45936.0,30376.0,64648.0,0,"[cube3: pushed fp synapses, pushed false posit..."
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,shiyan,606368,606368,neuron 606369,45968.0,30448.0,64656.0,0,"[cube3: pushed fp synapses, pushed false posit..."
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,shiyan,606372,606372,neuron 606373,47320.0,26240.0,65176.0,0,"[cube3: pushed fp synapses, pushed false posit..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,shiyan,612349,612349,neuron 612350,65600.0,51520.0,41920.0,0,"[pushed false positives synapses, cube1: pushe..."
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,shiyan,612353,612353,neuron 612354,65592.0,51400.0,41952.0,0,"[pushed false positives synapses, cube1: pushe..."
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,shiyan,612357,612357,neuron 612358,65744.0,51336.0,41928.0,0,"[pushed false positives synapses, cube1: pushe..."
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,shiyan,612361,612361,neuron 612362,67544.0,52096.0,41856.0,0,"[pushed false positives synapses, cube1: pushe..."


In [ ]:
# Figure out which cube it is and put it in a new column
# Define a function to extract cube information
def get_cube_info(annotations):
    if not isinstance(annotations, list):
        return 'unknown'
    for anno in annotations:
        if 'cube' in anno.lower():
            if 'cube1' in anno.lower():
                return 'cube1'
            elif 'cube2' in anno.lower():
                return 'cube2'
            elif 'cube3' in anno.lower():
                return 'cube3'
    return 'unknown'

# Add cube column
final_post_conn_df_after_trans_filt['cube'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_cube_info)
display(final_post_conn_df_after_trans_filt)

# Add other_annotations to new column
def get_other_annotations(annotations):
    if not isinstance(annotations, list):
        return []
    return [anno for anno in annotations 
            if 'cube' not in anno.lower() 
            and 'pushed false positives synapses' not in anno.lower()]

# Add other_annotations column
final_post_conn_df_after_trans_filt['other_annotations'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_other_annotations)

display(final_post_conn_df_after_trans_filt)

# save to csv
# final_post_conn_df_after_trans_filt.to_csv(f"{output_dir}/final_post_conn_df_after_trans_filt.csv", index=False)


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id,annotations,cube
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,shiyan,606348,606348,neuron 606349,45688.0,30680.0,64360.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,shiyan,606352,606352,neuron 606353,45792.0,30744.0,64320.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,shiyan,606364,606364,neuron 606365,45936.0,30376.0,64648.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,shiyan,606368,606368,neuron 606369,45968.0,30448.0,64656.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,shiyan,606372,606372,neuron 606373,47320.0,26240.0,65176.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,shiyan,612349,612349,neuron 612350,65600.0,51520.0,41920.0,0,"[pushed false positives synapses, cube1: pushe...",cube1
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,shiyan,612353,612353,neuron 612354,65592.0,51400.0,41952.0,0,"[pushed false positives synapses, cube1: pushe...",cube1
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,shiyan,612357,612357,neuron 612358,65744.0,51336.0,41928.0,0,"[pushed false positives synapses, cube1: pushe...",cube1
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,shiyan,612361,612361,neuron 612362,67544.0,52096.0,41856.0,0,"[pushed false positives synapses, cube1: pushe...",cube1


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,...,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id,annotations,cube,other_annotations
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,...,606348,606348,neuron 606349,45688.0,30680.0,64360.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, wrong set]"
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,...,606352,606352,neuron 606353,45792.0,30744.0,64320.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, wrong set]"
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,...,606364,606364,neuron 606365,45936.0,30376.0,64648.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, uncertain]"
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,...,606368,606368,neuron 606369,45968.0,30448.0,64656.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[distanced set, uncertain]"
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,...,606372,606372,neuron 606373,47320.0,26240.0,65176.0,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, wrong set]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,...,612349,612349,neuron 612350,65600.0,51520.0,41920.0,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: pre correct]
158,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,...,612353,612353,neuron 612354,65592.0,51400.0,41952.0,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: pre correct]
159,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,...,612357,612357,neuron 612358,65744.0,51336.0,41928.0,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: unknown]
160,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,...,612361,612361,neuron 612362,67544.0,52096.0,41856.0,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: pre correct]


In [ ]:
# Now link it back to the transactions in today_df_filt

# Group by user and label and count the number of transactions
grouped_total_df = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_total_df)
display(today_df_filt)


#  Merge based on matching coordinates
final_post_conn_df_after_trans_filt_merge = final_post_conn_df_after_trans_filt.merge(
    today_df_filt[['post_x', 'post_y', 'post_z', 'user', 'user_id', 'project_id', 'transaction_id', 'execution_time', 'label']],
    left_on=['post_x', 'post_y', 'post_z'],
    right_on=['post_x', 'post_y', 'post_z'],
    how='inner'
)

display(final_post_conn_df_after_trans_filt_merge)

# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user']).size().reset_index(name='counts')
display(grouped)

# save csv file
final_post_conn_df_after_trans_filt_merge.to_csv(f"{output_dir}/final_df_postsyn_transaction_octo.csv", index=False)



,user,label,counts
0,adulac,annotations.add,73
1,mclayton,annotations.add,53
2,mrobbins,annotations.add,18
3,sharris,annotations.add,29


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon,post_x,post_y,post_z
7,2504056,2025-03-06 19:15:08.061896+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104664.0, 'y': 53920.0, 'z': 32096.0}",Reviewer 6,104664.0,53920.0,32096.0
8,2504054,2025-03-06 19:10:09.244155+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102224.0, 'y': 52888.0, 'z': 32264.0}",Reviewer 6,102224.0,52888.0,32264.0
9,2504052,2025-03-06 19:07:56.060628+00:00,47,18,Backend,annotations.add,sharris,"{'x': 102064.0, 'y': 52872.0, 'z': 32280.0}",Reviewer 6,102064.0,52872.0,32280.0
10,2504050,2025-03-06 19:05:27.749166+00:00,47,18,Backend,annotations.add,sharris,"{'x': 104480.0, 'y': 52576.0, 'z': 32016.0}",Reviewer 6,104480.0,52576.0,32016.0
11,2504048,2025-03-06 19:04:12.100705+00:00,47,18,Backend,annotations.add,sharris,"{'x': 101040.0, 'y': 52128.0, 'z': 32576.0}",Reviewer 6,101040.0,52128.0,32576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
204,2502353,2025-03-06 16:42:47.417239+00:00,18,18,Backend,annotations.add,adulac,"{'x': 68456.0, 'y': 47440.0, 'z': 37912.0}",Reviewer 4,68456.0,47440.0,37912.0
209,2502318,2025-03-06 16:41:27.477313+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 45792.0, 'y': 30640.0, 'z': 65080.0}",Reviewer 2,45792.0,30640.0,65080.0
210,2502305,2025-03-06 16:40:45.982455+00:00,23,18,Backend,annotations.add,mclayton,"{'x': 69088.0, 'y': 47448.0, 'z': 38048.0}",Reviewer 1,69088.0,47448.0,38048.0
218,2502261,2025-03-06 16:38:18.985573+00:00,39,18,Backend,annotations.add,mrobbins,"{'x': 47384.0, 'y': 26144.0, 'z': 65248.0}",Reviewer 2,47384.0,26144.0,65248.0


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,...,node_id,annotations,cube,other_annotations,user,user_id,project_id,transaction_id,execution_time,label
0,2109930,45816.0,30632.0,64160.0,5,2025-02-20 14:13:46.706904,2025-02-20 14:13:46.706904,NaN,Postsynaptic,shiyan,...,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, wrong set]",mrobbins,39,18,2502857,2025-03-06 17:07:02.298032+00:00,annotations.add
1,2109931,45832.0,30608.0,64136.0,5,2025-02-20 14:13:46.801449,2025-02-20 14:13:46.801449,NaN,Postsynaptic,shiyan,...,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, wrong set]",mrobbins,39,18,2502850,2025-03-06 17:06:25.777110+00:00,annotations.add
2,2109934,45856.0,30400.0,64680.0,5,2025-02-20 14:13:47.077405,2025-02-20 14:13:47.077405,NaN,Postsynaptic,shiyan,...,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, uncertain]",mrobbins,39,18,2502824,2025-03-06 17:04:11.210055+00:00,annotations.add
3,2109935,45848.0,30424.0,64688.0,5,2025-02-20 14:13:47.170792,2025-02-20 14:13:47.170792,NaN,Postsynaptic,shiyan,...,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[distanced set, uncertain]",mrobbins,39,18,2502778,2025-03-06 17:02:49.551908+00:00,annotations.add
4,2109936,47416.0,26144.0,65208.0,5,2025-02-20 14:13:47.266040,2025-02-20 14:13:47.266040,NaN,Postsynaptic,shiyan,...,0,"[cube3: pushed fp synapses, pushed false posit...",cube3,"[pre correct, wrong set]",mrobbins,39,18,2502716,2025-03-06 17:01:09.552364+00:00,annotations.add
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,2119919,65712.0,51464.0,41952.0,5,2025-02-26 13:43:07.975596,2025-02-26 13:43:07.975596,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: pre correct],mclayton,23,18,2503255,2025-03-06 17:33:10.934784+00:00,annotations.add
168,2119920,65728.0,51480.0,41944.0,5,2025-02-26 13:43:08.068869,2025-02-26 13:43:08.068869,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: pre correct],mclayton,23,18,2503240,2025-03-06 17:32:45.278293+00:00,annotations.add
169,2119921,65728.0,51496.0,41944.0,5,2025-02-26 13:43:08.158772,2025-02-26 13:43:08.158772,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: unknown],mclayton,23,18,2503224,2025-03-06 17:32:16.267447+00:00,annotations.add
170,2119922,67432.0,52088.0,42040.0,5,2025-02-26 13:43:08.245989,2025-02-26 13:43:08.245989,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, cube1: pushe...",cube1,[mc: pre correct],mclayton,23,18,2503064,2025-03-06 17:26:32.020520+00:00,annotations.add


,user,counts
0,adulac,73
1,mclayton,52
2,mrobbins,18
3,sharris,29


In [ ]:
# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user', 'cube']).size().reset_index(name='counts')
display(grouped)

# find which users overlap in  'post_x', 'post_y', 'post_z'
# Find overlapping coordinates
grouped_overlap = final_post_conn_df_after_trans_filt_merge.groupby(['post_x', 'post_y', 'post_z']).size().reset_index(name='counts')
overlapping_coords = grouped_overlap[grouped_overlap['counts'] > 1]
display(overlapping_coords)


# Get the details for these overlapping coordinates
overlapping_details = final_post_conn_df_after_trans_filt_merge[
    final_post_conn_df_after_trans_filt_merge[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1).isin(
        overlapping_coords[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1)
    )
][['user', 'cube', 'post_x', 'post_y', 'post_z', 'other_annotations']]

# Display the results
display(overlapping_details.sort_values(['post_x', 'post_y', 'post_z']))

,user,cube,counts
0,adulac,cube1,73
1,mclayton,cube1,52
2,mrobbins,cube3,18
3,sharris,cube2,29


,post_x,post_y,post_z,counts
20,65160.0,49448.0,38008.0,2
38,66088.0,49392.0,38336.0,2
77,68184.0,50488.0,38048.0,2
83,68456.0,47440.0,37912.0,2
99,68896.0,51776.0,37816.0,3
103,69088.0,47448.0,38048.0,2
109,69208.0,51520.0,38072.0,2
121,69552.0,52088.0,38376.0,2
130,69784.0,48200.0,38080.0,2


,user,cube,post_x,post_y,post_z,other_annotations
62,adulac,cube1,65160.0,49448.0,38008.0,[pre correct]
63,mclayton,cube1,65160.0,49448.0,38008.0,[pre correct]
60,adulac,cube1,66088.0,49392.0,38336.0,[pre correct]
61,mclayton,cube1,66088.0,49392.0,38336.0,[pre correct]
67,mclayton,cube1,68184.0,50488.0,38048.0,"[wrong set, uncertain]"
68,adulac,cube1,68184.0,50488.0,38048.0,"[wrong set, uncertain]"
50,mclayton,cube1,68456.0,47440.0,37912.0,[distanced set]
51,adulac,cube1,68456.0,47440.0,37912.0,[distanced set]
70,adulac,cube1,68896.0,51776.0,37816.0,"[pre correct, distanced set, dist]"
71,mclayton,cube1,68896.0,51776.0,37816.0,"[pre correct, distanced set, dist]"


# MR1.4-3

In [190]:
rm_mr = pymaid.CatmaidInstance(volumes[0]["url"], project_id=volumes[0]["project"], api_token=volumes[0]["token"],
                               http_user=http_user, http_password=http_pass)

rm_mr


INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


CatmaidInstance at 6593919152.
Server: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem
Project: 19
Caching True (size limit 128; time limit None)
Cache size: 0.0

In [191]:
# Get transactions for MR143

project_id = volumes[1]["project"]

# Now call the function to retrieve enriched transactions.
try:
    txns_with_locations = get_transactions_with_locations(rm_mr, project_id, range_length=500)
    print(txns_with_locations)
    for txn in txns_with_locations:
        print("Transaction ID:", txn.get("transaction_id"))
        print("Location Data:", txn.get("location_data"))
        print("=" * 40)
except Exception as e:
    print("An error occurred:", e)

Error fetching location data for transaction 2505772: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/19/transactions/location?transaction_id=2505772&execution_time=2025-03-07+11%3A51%3A45.335754%2B00%3A00
Error fetching location data for transaction 2504193: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/19/transactions/location?transaction_id=2504193&execution_time=2025-03-06+21%3A59%3A11.890904%2B00%3A00
Error fetching location data for transaction 2504190: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/19/transactions/location?transaction_id=2504190&execution_time=2025-03-06+21%3A53%3A53.506639%2B00%3A00
Error fetching location data for transaction 2504187: 1 errors encountered: 400 Server Error: Bad Request for url: https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/19/transactions/loca

In [192]:
import pandas as pd
output_dir = "/Users/sam/Library/CloudStorage/OneDrive-UniversityofCambridge/Synapse_localisation/synapse_curation/catmaid_tracker_plots"

# Convert to DataFrame
df = pd.DataFrame(txns_with_locations)

# Print the DataFrame
display(df)

# Save the DataFrame to a CSV file
df.to_csv(f"{output_dir}/transactions_with_locations_mr143.csv", index=False)

# group by user and label and count the number of transactions
grouped = df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped

,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data
0,2507395,2025-03-07 15:21:44.866416+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68568.0, 'y': 57472.0, 'z': 104912.0}"
1,2507393,2025-03-07 15:20:48.481371+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68656.0, 'y': 57472.0, 'z': 104864.0}"
2,2507390,2025-03-07 15:19:49.086707+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68536.0, 'y': 57152.0, 'z': 104816.0}"
3,2507388,2025-03-07 15:19:16.470134+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69104.0, 'y': 57568.0, 'z': 105368.0}"
4,2507386,2025-03-07 15:17:51.156581+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69064.0, 'y': 57600.0, 'z': 105272.0}"
...,...,...,...,...,...,...,...,...
495,2500920,2025-03-05 14:02:44.067174+00:00,38,19,Backend,links.create,shiyan,"{'x': 71776.0, 'y': 50720.0, 'z': 75280.0}"
496,2500919,2025-03-05 14:02:44.064253+00:00,38,19,Backend,links.create,shiyan,"{'x': 71640.0, 'y': 50680.0, 'z': 75296.0}"
497,2500918,2025-03-05 14:02:44.010690+00:00,38,19,Backend,links.create,shiyan,"{'x': 71656.0, 'y': 50872.0, 'z': 75296.0}"
498,2500917,2025-03-05 14:02:43.999818+00:00,38,19,Backend,links.create,shiyan,"{'x': 71600.0, 'y': 48904.0, 'z': 76184.0}"


,user,label,counts
0,gmo,annotations.add,4
1,gmo,treenodes.create,1
2,gmo,treenodes.remove,1
3,hack_guest,treenodes.create,1
4,hack_guest,treenodes.remove,1
5,mrobbins,annotations.add,135
6,mrobbins,annotations.remove,2
7,nceffa,annotations.add,157
8,nceffa,annotations.remove,2
9,nceffa,labels.update,2


In [195]:
# Filter transactions on the hackathon day
import pandas as pd
from datetime import datetime


# Convert execution_time to datetime if it's not already
df['execution_time'] = pd.to_datetime(df['execution_time'])

# Get today's date
today = pd.Timestamp(2025, 3, 6).date() #datetime.now().date()
print(today)

# Filter for today's transactions
today_df = df[df['execution_time'].dt.date >= today]

# drop one user- smohinta-su
today_df = today_df[today_df['user'] != 'smohinta-su']
today_df = today_df[today_df['user'] != 'smohinta']


# replace hack_guest with sharris
today_df['user'] = today_df['user'].replace({'hack_guest': 'sharris'})

# replace realnames with anonymised reviewer names
today_df['user_anon'] = today_df['user'].replace({'mclayton': 'Reviewer 1', 'mrobbins': 'Reviewer 2', 'gmo': 'Reviewer 3', 'adulac': 'Reviewer 4',
                                             'nceffa': 'Reviewer 5',
                                             'sharris': 'Reviewer 6',
                                             'swilson': 'Reviewer 7',
                                             'shiyan': 'Reviewer 8',
                                             'hack_guest': 'Reviewer 9'
                                             }) 

display(today_df)


# group by user and label and count the number of transactions
grouped = today_df.groupby(['user', 'label']).size().reset_index(name='counts')
grouped # this looks correct now based on our expectations from talking to the reviewers


2025-03-06


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon
0,2507395,2025-03-07 15:21:44.866416+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68568.0, 'y': 57472.0, 'z': 104912.0}",Reviewer 5
1,2507393,2025-03-07 15:20:48.481371+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68656.0, 'y': 57472.0, 'z': 104864.0}",Reviewer 5
2,2507390,2025-03-07 15:19:49.086707+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68536.0, 'y': 57152.0, 'z': 104816.0}",Reviewer 5
3,2507388,2025-03-07 15:19:16.470134+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69104.0, 'y': 57568.0, 'z': 105368.0}",Reviewer 5
4,2507386,2025-03-07 15:17:51.156581+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69064.0, 'y': 57600.0, 'z': 105272.0}",Reviewer 5
...,...,...,...,...,...,...,...,...,...
359,2502583,2025-03-06 16:55:01.033072+00:00,46,19,Backend,annotations.add,gmo,"{'x': 71128.0, 'y': 49360.0, 'z': 74208.0}",Reviewer 3
360,2502572,2025-03-06 16:53:34.999663+00:00,46,19,Backend,annotations.add,gmo,"{'x': 70640.0, 'y': 49376.0, 'z': 74224.0}",Reviewer 3
361,2502561,2025-03-06 16:52:16.914798+00:00,46,19,Backend,annotations.add,gmo,"{'x': 71240.0, 'y': 49328.0, 'z': 74160.0}",Reviewer 3
362,2502535,2025-03-06 16:51:20.070166+00:00,46,19,Backend,treenodes.remove,gmo,"{'x': 71279.95, 'y': 49265.01, 'z': 74160.0}",Reviewer 3


,user,label,counts
0,gmo,annotations.add,4
1,gmo,treenodes.create,1
2,gmo,treenodes.remove,1
3,mrobbins,annotations.add,135
4,mrobbins,annotations.remove,2
5,nceffa,annotations.add,157
6,nceffa,annotations.remove,2
7,nceffa,labels.update,2
8,nceffa,links.create,2
9,nceffa,nodes.update_location,3


In [196]:
# Filter only annotations:add from transactions because we are interested in synapse annotations

today_df_filt = today_df[today_df['label'] == 'annotations.add']
display(today_df_filt)


# Again group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon
0,2507395,2025-03-07 15:21:44.866416+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68568.0, 'y': 57472.0, 'z': 104912.0}",Reviewer 5
1,2507393,2025-03-07 15:20:48.481371+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68656.0, 'y': 57472.0, 'z': 104864.0}",Reviewer 5
2,2507390,2025-03-07 15:19:49.086707+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68536.0, 'y': 57152.0, 'z': 104816.0}",Reviewer 5
3,2507388,2025-03-07 15:19:16.470134+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69104.0, 'y': 57568.0, 'z': 105368.0}",Reviewer 5
4,2507386,2025-03-07 15:17:51.156581+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69064.0, 'y': 57600.0, 'z': 105272.0}",Reviewer 5
...,...,...,...,...,...,...,...,...,...
355,2502893,2025-03-06 17:10:34.966931+00:00,39,19,Backend,annotations.add,mrobbins,"{'x': 66144.0, 'y': 58672.0, 'z': 105880.0}",Reviewer 2
358,2502605,2025-03-06 16:56:36.234267+00:00,46,19,Backend,annotations.add,gmo,"{'x': 69896.0, 'y': 49232.0, 'z': 74176.0}",Reviewer 3
359,2502583,2025-03-06 16:55:01.033072+00:00,46,19,Backend,annotations.add,gmo,"{'x': 71128.0, 'y': 49360.0, 'z': 74208.0}",Reviewer 3
360,2502572,2025-03-06 16:53:34.999663+00:00,46,19,Backend,annotations.add,gmo,"{'x': 70640.0, 'y': 49376.0, 'z': 74224.0}",Reviewer 3


,user,label,counts
0,gmo,annotations.add,4
1,mrobbins,annotations.add,135
2,nceffa,annotations.add,157
3,shiyan,annotations.add,24
4,swilson,annotations.add,7


In [198]:
# Save the filtered DataFrame to a CSV file
today_df_filt.to_csv(f"{output_dir}/transactions_with_locations_filtered_mr143.csv", index=False)

In [199]:
# Check how many location_data = {'x': None, 'y': None, 'z': None} are there
display(today_df_filt['location_data'].value_counts())

# Which users have location_data = {'x': None, 'y': None, 'z': None}
location_missing = today_df_filt[today_df_filt['location_data'] == {'x': None, 'y': None, 'z': None}]
# group by user and label and count the number of transactions
grouped_missing = location_missing.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_missing)


# Filter out all missing locations
today_df_filt = today_df_filt[today_df_filt['location_data'] != {'x': None, 'y': None, 'z': None}]
display(today_df_filt)

# Extract x, y, z coordinates into separate columns
today_df_filt['post_x'] = today_df_filt['location_data'].apply(lambda x: x['x'])
today_df_filt['post_y'] = today_df_filt['location_data'].apply(lambda x: x['y'])
today_df_filt['post_z'] = today_df_filt['location_data'].apply(lambda x: x['z'])
display(today_df_filt)


# Group by user and label and count the number of transactions
grouped_filt = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
grouped_filt


location_data
{'x': None, 'y': None, 'z': None}              13
{'x': 67024.0, 'y': 56832.0, 'z': 103096.0}     5
{'x': 66032.0, 'y': 56056.0, 'z': 104608.0}     4
{'x': 68776.0, 'y': 55856.0, 'z': 102568.0}     3
{'x': 68640.0, 'y': 55544.0, 'z': 102704.0}     3
                                               ..
{'x': 66648.0, 'y': 56040.0, 'z': 103968.0}     1
{'x': 67832.0, 'y': 56384.0, 'z': 103328.0}     1
{'x': 67856.0, 'y': 56304.0, 'z': 103264.0}     1
{'x': 67864.0, 'y': 56488.0, 'z': 103864.0}     1
{'x': 70640.0, 'y': 49376.0, 'z': 74224.0}      1
Name: count, Length: 269, dtype: int64

,user,label,counts
0,mrobbins,annotations.add,2
1,nceffa,annotations.add,11


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon
0,2507395,2025-03-07 15:21:44.866416+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68568.0, 'y': 57472.0, 'z': 104912.0}",Reviewer 5
1,2507393,2025-03-07 15:20:48.481371+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68656.0, 'y': 57472.0, 'z': 104864.0}",Reviewer 5
2,2507390,2025-03-07 15:19:49.086707+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68536.0, 'y': 57152.0, 'z': 104816.0}",Reviewer 5
3,2507388,2025-03-07 15:19:16.470134+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69104.0, 'y': 57568.0, 'z': 105368.0}",Reviewer 5
4,2507386,2025-03-07 15:17:51.156581+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69064.0, 'y': 57600.0, 'z': 105272.0}",Reviewer 5
...,...,...,...,...,...,...,...,...,...
355,2502893,2025-03-06 17:10:34.966931+00:00,39,19,Backend,annotations.add,mrobbins,"{'x': 66144.0, 'y': 58672.0, 'z': 105880.0}",Reviewer 2
358,2502605,2025-03-06 16:56:36.234267+00:00,46,19,Backend,annotations.add,gmo,"{'x': 69896.0, 'y': 49232.0, 'z': 74176.0}",Reviewer 3
359,2502583,2025-03-06 16:55:01.033072+00:00,46,19,Backend,annotations.add,gmo,"{'x': 71128.0, 'y': 49360.0, 'z': 74208.0}",Reviewer 3
360,2502572,2025-03-06 16:53:34.999663+00:00,46,19,Backend,annotations.add,gmo,"{'x': 70640.0, 'y': 49376.0, 'z': 74224.0}",Reviewer 3


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon,post_x,post_y,post_z
0,2507395,2025-03-07 15:21:44.866416+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68568.0, 'y': 57472.0, 'z': 104912.0}",Reviewer 5,68568.0,57472.0,104912.0
1,2507393,2025-03-07 15:20:48.481371+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68656.0, 'y': 57472.0, 'z': 104864.0}",Reviewer 5,68656.0,57472.0,104864.0
2,2507390,2025-03-07 15:19:49.086707+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68536.0, 'y': 57152.0, 'z': 104816.0}",Reviewer 5,68536.0,57152.0,104816.0
3,2507388,2025-03-07 15:19:16.470134+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69104.0, 'y': 57568.0, 'z': 105368.0}",Reviewer 5,69104.0,57568.0,105368.0
4,2507386,2025-03-07 15:17:51.156581+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69064.0, 'y': 57600.0, 'z': 105272.0}",Reviewer 5,69064.0,57600.0,105272.0
...,...,...,...,...,...,...,...,...,...,...,...,...
355,2502893,2025-03-06 17:10:34.966931+00:00,39,19,Backend,annotations.add,mrobbins,"{'x': 66144.0, 'y': 58672.0, 'z': 105880.0}",Reviewer 2,66144.0,58672.0,105880.0
358,2502605,2025-03-06 16:56:36.234267+00:00,46,19,Backend,annotations.add,gmo,"{'x': 69896.0, 'y': 49232.0, 'z': 74176.0}",Reviewer 3,69896.0,49232.0,74176.0
359,2502583,2025-03-06 16:55:01.033072+00:00,46,19,Backend,annotations.add,gmo,"{'x': 71128.0, 'y': 49360.0, 'z': 74208.0}",Reviewer 3,71128.0,49360.0,74208.0
360,2502572,2025-03-06 16:53:34.999663+00:00,46,19,Backend,annotations.add,gmo,"{'x': 70640.0, 'y': 49376.0, 'z': 74224.0}",Reviewer 3,70640.0,49376.0,74224.0


,user,label,counts
0,gmo,annotations.add,4
1,mrobbins,annotations.add,133
2,nceffa,annotations.add,146
3,shiyan,annotations.add,24
4,swilson,annotations.add,7


#### Find all locations based on annotations list of MR143

In [201]:
# Find all neurons_ids and locations based on annotations list
import pandas as pd

annotations_list_mr143 = ["annotation:cube1MBONj2: pushed fp synapses", "annotation:cube2likelyj2: pushed fp synapses"]

def get_neurons_from_annotations_list(annotations_list, remote_instance):
    """
    Get all neurons and their locations based on a list of annotations.
    """
    neurons = []
    for annotation in annotations_list:
        # Get all annotations with the given name
        annotation_neurons = pymaid.get_neurons(annotation, remote_instance=remote_instance)
        neurons.extend(annotation_neurons)
    return neurons

neurons = get_neurons_from_annotations_list(annotations_list_mr143, rm_mr)
print(neurons)

# Assuming these are all post neurons 
post = [neuron.id for neuron in neurons]
for i, neuron in enumerate(neurons):
    node_coords = neuron.nodes[['x', 'y', 'z']]
    print(f"Coordinates for neuron {neuron.skeleton_id}:")
    print(node_coords)
    print("=" * 40)
    if i == 2:
        break


# Create a list to store all neuron data
neuron_data = []

for neuron in neurons:
    # Get coordinates for each node in the neuron
    node_coords = neuron.nodes[['x', 'y', 'z']]
    
    # For each node in the neuron, create a row with all information
    for idx, coords in node_coords.iterrows():
        neuron_data.append({
            'neuron_id': neuron.id,
            'skeleton_id': neuron.skeleton_id,
            'name': neuron.name,
            'x': coords['x'],
            'y': coords['y'],
            'z': coords['z'],
            'node_id': idx
        })

# Convert to DataFrame
post_neurons_all_from_ann_df = pd.DataFrame(neuron_data)
display(post_neurons_all_from_ann_df)

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


Fetch neurons:   0%|          | 0/181 [00:00<?, ?it/s]

Make nrn:   0%|          | 0/181 [00:00<?, ?it/s]

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


Fetch neurons:   0%|          | 0/193 [00:00<?, ?it/s]

Make nrn:   0%|          | 0/193 [00:00<?, ?it/s]

[type            CatmaidNeuron
name            neuron 620546
id                     620545
n_nodes                     1
n_connectors                1
n_branches                  0
n_leafs                     0
cable_length              0.0
soma                     None
units             1 nanometer
dtype: object, type            CatmaidNeuron
name            neuron 621058
id                     621057
n_nodes                     1
n_connectors                1
n_branches                  0
n_leafs                     0
cable_length              0.0
soma                     None
units             1 nanometer
dtype: object, type            CatmaidNeuron
name            neuron 620550
id                     620549
n_nodes                     1
n_connectors                1
n_branches                  0
n_leafs                     0
cable_length              0.0
soma                     None
units             1 nanometer
dtype: object, type            CatmaidNeuron
name            neuron 6

,neuron_id,skeleton_id,name,x,y,z,node_id
0,620545,620545,neuron 620546,67024.0,56832.0,103096.0,0
1,621057,621057,neuron 621058,67560.0,57296.0,105208.0,0
2,620549,620549,neuron 620550,66968.0,58680.0,102864.0,0
3,621061,621061,neuron 621062,68824.0,58184.0,105448.0,0
4,620553,620553,neuron 620554,66952.0,58584.0,103000.0,0
...,...,...,...,...,...,...,...
372,622060,622060,neuron 622061,71400.0,51168.0,74984.0,0
373,622064,622064,neuron 622065,71288.0,51120.0,74992.0,0
374,622068,622068,neuron 622069,71208.0,51216.0,75064.0,0
375,622072,622072,neuron 622073,71368.0,51328.0,75056.0,0


In [202]:
# Filter the post_neurons_all_from_ann_df based on the post neurons we have from today_df_filt based on locations
# Create a mask for matching coordinates
matching_coords = post_neurons_all_from_ann_df.apply(
    lambda row: any((row['x'] == today_df_filt['post_x']) & 
                   (row['y'] == today_df_filt['post_y']) & 
                   (row['z'] == today_df_filt['post_z'])), 
    axis=1
)

# Filter post_neurons_all_from_ann_df to keep only the matching rows
post_neurons_all_from_ann_df_filterby_today_df = post_neurons_all_from_ann_df[matching_coords]
display(post_neurons_all_from_ann_df_filterby_today_df)


,neuron_id,skeleton_id,name,x,y,z,node_id
0,620545,620545,neuron 620546,67024.0,56832.0,103096.0,0
2,620549,620549,neuron 620550,66968.0,58680.0,102864.0,0
4,620553,620553,neuron 620554,66952.0,58584.0,103000.0,0
8,620561,620561,neuron 620562,68904.0,55560.0,103152.0,0
12,620569,620569,neuron 620570,68400.0,55800.0,103816.0,0
...,...,...,...,...,...,...,...
372,622060,622060,neuron 622061,71400.0,51168.0,74984.0,0
373,622064,622064,neuron 622065,71288.0,51120.0,74992.0,0
374,622068,622068,neuron 622069,71208.0,51216.0,75064.0,0
375,622072,622072,neuron 622073,71368.0,51328.0,75056.0,0


In [205]:
post = [neuron.skeleton_id for i, neuron in post_neurons_all_from_ann_df_filterby_today_df.iterrows()]

# Get all connections for the neurons
connections = pymaid.get_connectors(post, remote_instance=rm_mr)
display(connections)

connector_ids = connections['connector_id'].tolist()
print(connector_ids)
connector_details = pymaid.get_connector_details(connector_ids)
display(connector_details)

# Step 2: Rename `postsynaptic_to` to `skeleton_id` for clarity
connector_details = connector_details.rename(columns={'postsynaptic_to': 'skeleton_id'})

# Step 3: Convert skeleton_id to numeric type. Note this cannot work when > 1 skeleton_id is in list
# Extract the first (and presumably only) element from each list in the 'postsynaptic_to' column
connector_details['skeleton_id'] = connector_details['skeleton_id'].apply(lambda x: x[0] if x else None)

connector_details['skeleton_id'] = connector_details['skeleton_id'].astype(int)

# Step 4: Merge with connectors DataFrame (if needed)
connectors_with_skid = pd.merge(connections, connector_details[['connector_id', 'skeleton_id']], 
                                on='connector_id', how='left')
display(connectors_with_skid)

print(f"dtype connectors_with_skid['skeleton_id'].dtype: {connectors_with_skid['skeleton_id'].dtype}")
print(f"post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype: {post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype}")

# Convert to numeric type
post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'] = post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].astype(int)

# Step 5: Merge with post_neurons_df. Note connector = pre
final_post_conn_df_after_trans_filt = pd.merge(connectors_with_skid, post_neurons_all_from_ann_df_filterby_today_df, 
                  on='skeleton_id', how='left', suffixes=('_connector', '_post'))

display(final_post_conn_df_after_trans_filt)


final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.rename(columns={
    'x_connector': 'connector_x',
    'y_connector': 'connector_y',
    'z_connector': 'connector_z',
    'x_post': 'post_x',
    'y_post': 'post_y',
    'z_post': 'post_z'
})
print("result")
display(final_post_conn_df_after_trans_filt)


INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


,connector_id,x,y,z,confidence,creation_time,edition_time,tags,type,creator,editor
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,shiyan
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,shiyan
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,shiyan
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,shiyan
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,shiyan
...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,shiyan
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,shiyan
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,shiyan
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,shiyan


[2125008, 2125009, 2125010, 2125011, 2125012, 2125013, 2125014, 2125015, 2125016, 2125017, 2125019, 2125020, 2125021, 2125022, 2125023, 2125024, 2125025, 2125026, 2125027, 2125028, 2125029, 2125030, 2125031, 2125032, 2125033, 2125035, 2125037, 2125038, 2125039, 2125040, 2125041, 2125042, 2125043, 2125044, 2125045, 2125046, 2125047, 2125048, 2125049, 2125050, 2125051, 2125052, 2125053, 2125054, 2125055, 2125056, 2125057, 2125058, 2125059, 2125060, 2125061, 2125062, 2125063, 2125064, 2125065, 2125066, 2125067, 2125068, 2125069, 2125070, 2125071, 2125072, 2125073, 2125074, 2125075, 2125076, 2125077, 2125078, 2125079, 2125080, 2125081, 2125082, 2125083, 2125084, 2125085, 2125086, 2125087, 2125088, 2125089, 2125090, 2125091, 2125092, 2125093, 2125094, 2125095, 2125096, 2125097, 2125098, 2125099, 2125100, 2125101, 2125102, 2125103, 2125104, 2125105, 2125106, 2125109, 2125110, 2125111, 2125112, 2125114, 2125115, 2125116, 2125117, 2125118, 2125119, 2125120, 2125121, 2125122, 2125123, 2125124, 

CN details:   0%|          | 0/264 [00:00<?, ?it/s]

INFO  : Data for 264 of 264 unique connector IDs retrieved (pymaid)


...

,connector_id,x,y,z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,shiyan,620449
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,shiyan,620453
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,shiyan,620457
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,shiyan,620461
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,shiyan,620465
...,...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,shiyan,622504
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,shiyan,622508
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,shiyan,622512
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,shiyan,622516


dtype connectors_with_skid['skeleton_id'].dtype: int64
post_neurons_all_from_ann_df_filterby_today_df['skeleton_id'].dtype: object


,connector_id,x_connector,y_connector,z_connector,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,x_post,y_post,z_post,node_id
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,shiyan,620449,620449,neuron 620450,68776.0,55856.0,102568.0,3
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,shiyan,620453,620453,neuron 620454,68744.0,55776.0,102616.0,0
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,shiyan,620457,620457,neuron 620458,68896.0,55840.0,102720.0,0
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,shiyan,620461,620461,neuron 620462,68960.0,55952.0,102952.0,0
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,shiyan,620465,620465,neuron 620466,68920.0,55552.0,103144.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,shiyan,622504,622504,neuron 622505,69416.0,50920.0,77128.0,0
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,shiyan,622508,622508,neuron 622509,71712.0,51080.0,77344.0,0
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,shiyan,622512,622512,neuron 622513,71120.0,51152.0,77200.0,0
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,shiyan,622516,622516,neuron 622517,71160.0,51328.0,77208.0,0


result


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,shiyan,620449,620449,neuron 620450,68776.0,55856.0,102568.0,3
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,shiyan,620453,620453,neuron 620454,68744.0,55776.0,102616.0,0
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,shiyan,620457,620457,neuron 620458,68896.0,55840.0,102720.0,0
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,shiyan,620461,620461,neuron 620462,68960.0,55952.0,102952.0,0
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,shiyan,620465,620465,neuron 620466,68920.0,55552.0,103144.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,shiyan,622504,622504,neuron 622505,69416.0,50920.0,77128.0,0
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,shiyan,622508,622508,neuron 622509,71712.0,51080.0,77344.0,0
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,shiyan,622512,622512,neuron 622513,71120.0,51152.0,77200.0,0
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,shiyan,622516,622516,neuron 622517,71160.0,51328.0,77208.0,0


In [206]:
# Assuming you have a DataFrame 'final_post_conn_df_after_trans_filt' with 'skeleton_id' columns
skeleton_ids = final_post_conn_df_after_trans_filt['skeleton_id'].tolist()

# Get annotations for skeletons
skeleton_annotations = pymaid.get_annotations(skeleton_ids, remote_instance=rm_mr)
print(skeleton_annotations)


# Assuming your annotations are in a dictionary called 'annotations'
def get_relevant_annotations(anno_list):
    return [a for a in anno_list if a not in ['cube2: pushed fp synapses', 'pushed false positives synapses', 
                                              'cube1 : pushed fp synapses', 'cube3: pushed fp synapses']]

# Convert annotations dictionary to DataFrame
anno_data = []
for skeleton_id, annotations in skeleton_annotations.items():
    anno_data.append({
        'skeleton_id': skeleton_id,
        'annotations': annotations
    })

anno_df = pd.DataFrame(anno_data)
display(anno_df)

# Convert skeleton_id to integer
anno_df['skeleton_id'] = anno_df['skeleton_id'].astype(int)

# Check number of rows before merge
print("Rows in final_post_conn_df_after_trans_filt before merge:", len(final_post_conn_df_after_trans_filt))
print("Rows in anno_df:", len(anno_df))

# Check for duplicate skeleton_ids in both DataFrames
print("\nDuplicate skeleton_ids in final_post_conn_df_after_trans_filt:")
print(final_post_conn_df_after_trans_filt['skeleton_id'].value_counts())
print("\nDuplicate skeleton_ids in anno_df:")
print(anno_df['skeleton_id'].value_counts())


# Add annotations to the `final_post_conn_df_after_trans_filt` DataFrame
final_post_conn_df_after_trans_filt = final_post_conn_df_after_trans_filt.merge(anno_df, on='skeleton_id', how='inner')

display(final_post_conn_df_after_trans_filt)

{'620477': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses', 'sy distanced'], '620509': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses', 'sy wrong'], '620513': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses', 'sy wrong'], '620533': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses'], '620545': ['ngc: Wrong Set', 'ngc: Uncertain', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses', 'ngc: Pre Correct', 'ngc: Distanced Set', 'sy distanced'], '620585': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses'], '620609': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses'], '620633': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushed fp synapses'], '620637': ['ngc: Wrong Set', 'pushed false positives synapses', 'cube1MBONj2: pushe

,skeleton_id,annotations
0,620477,"[ngc: Wrong Set, pushed false positives synaps..."
1,620509,"[ngc: Wrong Set, pushed false positives synaps..."
2,620513,"[ngc: Wrong Set, pushed false positives synaps..."
3,620533,"[ngc: Wrong Set, pushed false positives synaps..."
4,620545,"[ngc: Wrong Set, ngc: Uncertain, pushed false ..."
...,...,...
259,622504,"[pushed false positives synapses, correct, MR,..."
260,622508,"[pushed false positives synapses, correct, MR,..."
261,622512,"[pushed false positives synapses, correct, MR,..."
262,622516,"[pushed false positives synapses, correct, MR,..."


Rows in final_post_conn_df_after_trans_filt before merge: 264
Rows in anno_df: 264

Duplicate skeleton_ids in final_post_conn_df_after_trans_filt:
skeleton_id
620449    1
622192    1
622136    1
622140    1
622144    1
         ..
620841    1
620845    1
620849    1
620861    1
622520    1
Name: count, Length: 264, dtype: int64

Duplicate skeleton_ids in anno_df:
skeleton_id
620477    1
622192    1
622136    1
622140    1
622144    1
         ..
621149    1
620969    1
620965    1
620961    1
622520    1
Name: count, Length: 264, dtype: int64


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,editor,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id,annotations
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,shiyan,620449,620449,neuron 620450,68776.0,55856.0,102568.0,3,"[pushed false positives synapses, cube1MBONj2:..."
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,shiyan,620453,620453,neuron 620454,68744.0,55776.0,102616.0,0,"[pushed false positives synapses, cube1MBONj2:..."
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,shiyan,620457,620457,neuron 620458,68896.0,55840.0,102720.0,0,"[pushed false positives synapses, cube1MBONj2:..."
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,shiyan,620461,620461,neuron 620462,68960.0,55952.0,102952.0,0,"[pushed false positives synapses, cube1MBONj2:..."
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,shiyan,620465,620465,neuron 620466,68920.0,55552.0,103144.0,0,"[pushed false positives synapses, cube1MBONj2:..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,shiyan,622504,622504,neuron 622505,69416.0,50920.0,77128.0,0,"[pushed false positives synapses, correct, MR,..."
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,shiyan,622508,622508,neuron 622509,71712.0,51080.0,77344.0,0,"[pushed false positives synapses, correct, MR,..."
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,shiyan,622512,622512,neuron 622513,71120.0,51152.0,77200.0,0,"[pushed false positives synapses, correct, MR,..."
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,shiyan,622516,622516,neuron 622517,71160.0,51328.0,77208.0,0,"[pushed false positives synapses, correct, MR,..."


In [ ]:
# Figure out which cube it is and put it in a new column
# Define a function to extract cube information
def get_cube_info(annotations):
    if not isinstance(annotations, list):
        return 'unknown'
    for anno in annotations:
        if 'cube' in anno.lower():
            if 'cube1' in anno.lower():
                return 'cube1'
            elif 'cube2' in anno.lower():
                return 'cube2'
            elif 'cube3' in anno.lower():
                return 'cube3'
    return 'unknown'

# Add cube column
final_post_conn_df_after_trans_filt['cube'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_cube_info)
display(final_post_conn_df_after_trans_filt)

# Add other_annotations to new column
def get_other_annotations(annotations):
    if not isinstance(annotations, list):
        return []
    return [anno for anno in annotations 
            if 'cube' not in anno.lower() 
            and 'pushed false positives synapses' not in anno.lower()]

# Add other_annotations column
final_post_conn_df_after_trans_filt['other_annotations'] = final_post_conn_df_after_trans_filt['annotations'].apply(get_other_annotations)

display(final_post_conn_df_after_trans_filt)

# save to csv
# final_post_conn_df_after_trans_filt.to_csv(f"{output_dir}/final_post_conn_df_after_trans_filt_mr143.csv", index=False)


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,...,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id,annotations,cube,other_annotations
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,...,620449,620449,neuron 620450,68776.0,55856.0,102568.0,3,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[ngc: Pre Correct, sw correct, sy correct]"
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,...,620453,620453,neuron 620454,68744.0,55776.0,102616.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw uncertain, sy wrong, ngc: Distanced Set]"
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,...,620457,620457,neuron 620458,68896.0,55840.0,102720.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw distance, ngc: Pre Correct, sy correct]"
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,...,620461,620461,neuron 620462,68960.0,55952.0,102952.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw uncertain, ngc: Distanced Set, sy distanced]"
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,...,620465,620465,neuron 620466,68920.0,55552.0,103144.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw distance, ngc: Distanced Set, sy distanced]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,...,622504,622504,neuron 622505,69416.0,50920.0,77128.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,...,622508,622508,neuron 622509,71712.0,51080.0,77344.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,...,622512,622512,neuron 622513,71120.0,51152.0,77200.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,...,622516,622516,neuron 622517,71160.0,51328.0,77208.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,...,skeleton_id,neuron_id,name,post_x,post_y,post_z,node_id,annotations,cube,other_annotations
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,...,620449,620449,neuron 620450,68776.0,55856.0,102568.0,3,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[ngc: Pre Correct, sw correct, sy correct]"
1,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,...,620453,620453,neuron 620454,68744.0,55776.0,102616.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw uncertain, sy wrong, ngc: Distanced Set]"
2,2125010,68760.0,55920.0,102728.0,5,2025-03-05 11:44:53.444492,2025-03-05 11:44:53.444492,NaN,Postsynaptic,shiyan,...,620457,620457,neuron 620458,68896.0,55840.0,102720.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw distance, ngc: Pre Correct, sy correct]"
3,2125011,68920.0,55992.0,102920.0,5,2025-03-05 11:44:53.529715,2025-03-05 11:44:53.529715,NaN,Postsynaptic,shiyan,...,620461,620461,neuron 620462,68960.0,55952.0,102952.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw uncertain, ngc: Distanced Set, sy distanced]"
4,2125012,68840.0,55536.0,103232.0,5,2025-03-05 11:44:53.618438,2025-03-05 11:44:53.618438,NaN,Postsynaptic,shiyan,...,620465,620465,neuron 620466,68920.0,55552.0,103144.0,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw distance, ngc: Distanced Set, sy distanced]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,2125576,69400.0,50760.0,77024.0,5,2025-03-05 14:00:58.316399,2025-03-05 14:00:58.316399,NaN,Postsynaptic,shiyan,...,622504,622504,neuron 622505,69416.0,50920.0,77128.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"
260,2125577,71736.0,51232.0,77328.0,5,2025-03-05 14:00:58.411279,2025-03-05 14:00:58.411279,NaN,Postsynaptic,shiyan,...,622508,622508,neuron 622509,71712.0,51080.0,77344.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"
261,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,...,622512,622512,neuron 622513,71120.0,51152.0,77200.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"
262,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,...,622516,622516,neuron 622517,71160.0,51328.0,77208.0,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]"


In [ ]:
# Now link it back to the transactions in today_df_filt

# Group by user and label and count the number of transactions
grouped_total_df = today_df_filt.groupby(['user', 'label']).size().reset_index(name='counts')
display(grouped_total_df)
display(today_df_filt)


#  Merge based on matching coordinates
final_post_conn_df_after_trans_filt_merge = final_post_conn_df_after_trans_filt.merge(
    today_df_filt[['post_x', 'post_y', 'post_z', 'user', 'user_id', 'project_id', 'transaction_id', 'execution_time', 'label']],
    left_on=['post_x', 'post_y', 'post_z'],
    right_on=['post_x', 'post_y', 'post_z'],
    how='inner'
)

display(final_post_conn_df_after_trans_filt_merge)

# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user']).size().reset_index(name='counts')
display(grouped)

# save csv file
# final_post_conn_df_after_trans_filt_merge.to_csv(f"{output_dir}/final_df_postsyn_transaction_mr143.csv", index=False)



,user,label,counts
0,gmo,annotations.add,4
1,mrobbins,annotations.add,133
2,nceffa,annotations.add,146
3,shiyan,annotations.add,24
4,swilson,annotations.add,7


,transaction_id,execution_time,user_id,project_id,change_type,label,user,location_data,user_anon,post_x,post_y,post_z
0,2507395,2025-03-07 15:21:44.866416+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68568.0, 'y': 57472.0, 'z': 104912.0}",Reviewer 5,68568.0,57472.0,104912.0
1,2507393,2025-03-07 15:20:48.481371+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68656.0, 'y': 57472.0, 'z': 104864.0}",Reviewer 5,68656.0,57472.0,104864.0
2,2507390,2025-03-07 15:19:49.086707+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 68536.0, 'y': 57152.0, 'z': 104816.0}",Reviewer 5,68536.0,57152.0,104816.0
3,2507388,2025-03-07 15:19:16.470134+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69104.0, 'y': 57568.0, 'z': 105368.0}",Reviewer 5,69104.0,57568.0,105368.0
4,2507386,2025-03-07 15:17:51.156581+00:00,26,19,Backend,annotations.add,nceffa,"{'x': 69064.0, 'y': 57600.0, 'z': 105272.0}",Reviewer 5,69064.0,57600.0,105272.0
...,...,...,...,...,...,...,...,...,...,...,...,...
355,2502893,2025-03-06 17:10:34.966931+00:00,39,19,Backend,annotations.add,mrobbins,"{'x': 66144.0, 'y': 58672.0, 'z': 105880.0}",Reviewer 2,66144.0,58672.0,105880.0
358,2502605,2025-03-06 16:56:36.234267+00:00,46,19,Backend,annotations.add,gmo,"{'x': 69896.0, 'y': 49232.0, 'z': 74176.0}",Reviewer 3,69896.0,49232.0,74176.0
359,2502583,2025-03-06 16:55:01.033072+00:00,46,19,Backend,annotations.add,gmo,"{'x': 71128.0, 'y': 49360.0, 'z': 74208.0}",Reviewer 3,71128.0,49360.0,74208.0
360,2502572,2025-03-06 16:53:34.999663+00:00,46,19,Backend,annotations.add,gmo,"{'x': 70640.0, 'y': 49376.0, 'z': 74224.0}",Reviewer 3,70640.0,49376.0,74224.0


,connector_id,connector_x,connector_y,connector_z,confidence,creation_time,edition_time,tags,type,creator,...,node_id,annotations,cube,other_annotations,user,user_id,project_id,transaction_id,execution_time,label
0,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,...,3,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[ngc: Pre Correct, sw correct, sy correct]",nceffa,26,19,2503288,2025-03-06 17:34:56.336643+00:00,annotations.add
1,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,...,3,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[ngc: Pre Correct, sw correct, sy correct]",shiyan,38,19,2503075,2025-03-06 17:26:52.918870+00:00,annotations.add
2,2125008,68760.0,55896.0,102728.0,5,2025-03-05 11:44:53.264506,2025-03-05 11:44:53.264506,NaN,Postsynaptic,shiyan,...,3,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[ngc: Pre Correct, sw correct, sy correct]",swilson,44,19,2503039,2025-03-06 17:25:09.371431+00:00,annotations.add
3,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw uncertain, sy wrong, ngc: Distanced Set]",nceffa,26,19,2503401,2025-03-06 17:41:03.872753+00:00,annotations.add
4,2125009,68760.0,55928.0,102712.0,5,2025-03-05 11:44:53.352034,2025-03-05 11:44:53.352034,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, cube1MBONj2:...",cube1,"[sw uncertain, sy wrong, ngc: Distanced Set]",shiyan,38,19,2503353,2025-03-06 17:37:58.009333+00:00,annotations.add
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304,2125578,71200.0,51296.0,77208.0,5,2025-03-05 14:00:58.497052,2025-03-05 14:00:58.497052,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]",mrobbins,39,19,2502961,2025-03-06 17:18:42.437863+00:00,annotations.add
305,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]",mrobbins,39,19,2502987,2025-03-06 17:23:16.979202+00:00,annotations.add
306,2125579,71192.0,51240.0,77208.0,5,2025-03-05 14:00:58.585008,2025-03-05 14:00:58.585008,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, correct, MR,...",cube2,"[correct, MR, distanced]",mrobbins,39,19,2502959,2025-03-06 17:18:24.544699+00:00,annotations.add
307,2125580,69760.0,51632.0,77360.0,5,2025-03-05 14:00:58.677390,2025-03-05 14:00:58.677390,NaN,Postsynaptic,shiyan,...,0,"[pushed false positives synapses, wrong set, M...",cube2,"[wrong set, MR, distanced]",mrobbins,39,19,2502982,2025-03-06 17:22:56.384594+00:00,annotations.add


,user,counts
0,gmo,4
1,mrobbins,133
2,nceffa,142
3,shiyan,23
4,swilson,7


In [210]:
# group by user and label and count the number of transactions
grouped = final_post_conn_df_after_trans_filt_merge.groupby(['user', 'cube']).size().reset_index(name='counts')
display(grouped)

# find which users overlap in  'post_x', 'post_y', 'post_z'
# Find overlapping coordinates
grouped_overlap = final_post_conn_df_after_trans_filt_merge.groupby(['post_x', 'post_y', 'post_z']).size().reset_index(name='counts')
overlapping_coords = grouped_overlap[grouped_overlap['counts'] > 1]
display(overlapping_coords)


# Get the details for these overlapping coordinates
overlapping_details = final_post_conn_df_after_trans_filt_merge[
    final_post_conn_df_after_trans_filt_merge[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1).isin(
        overlapping_coords[['post_x', 'post_y', 'post_z']].apply(tuple, axis=1)
    )
][['user', 'cube', 'post_x', 'post_y', 'post_z', 'other_annotations']]

# Display the results
display(overlapping_details.sort_values(['post_x', 'post_y', 'post_z']))

,user,cube,counts
0,gmo,cube2,4
1,mrobbins,cube1,6
2,mrobbins,cube2,127
3,nceffa,cube1,142
4,shiyan,cube1,22
5,shiyan,cube2,1
6,swilson,cube1,7


,post_x,post_y,post_z,counts
6,66032.0,56056.0,104608.0,4
20,66392.0,55968.0,102792.0,2
21,66496.0,55824.0,102824.0,2
31,66872.0,55632.0,102952.0,2
33,66912.0,55720.0,102984.0,2
39,66968.0,56824.0,103136.0,2
43,67024.0,56832.0,103096.0,5
51,67136.0,56248.0,102808.0,2
57,67216.0,56072.0,102912.0,2
71,67608.0,56944.0,102928.0,2


,user,cube,post_x,post_y,post_z,other_annotations
126,nceffa,cube1,66032.0,56056.0,104608.0,"[ngc: Wrong Set, ngc: Uncertain, ngc: Pre Corr..."
127,nceffa,cube1,66032.0,56056.0,104608.0,"[ngc: Wrong Set, ngc: Uncertain, ngc: Pre Corr..."
128,nceffa,cube1,66032.0,56056.0,104608.0,"[ngc: Wrong Set, ngc: Uncertain, ngc: Pre Corr..."
129,nceffa,cube1,66032.0,56056.0,104608.0,"[ngc: Wrong Set, ngc: Uncertain, ngc: Pre Corr..."
21,nceffa,cube1,66392.0,55968.0,102792.0,"[ngc: Wrong Set, sy distanced]"
...,...,...,...,...,...,...
185,mrobbins,cube2,71400.0,51168.0,74984.0,"[wrong set, MR, distanced]"
301,mrobbins,cube2,71712.0,51080.0,77344.0,"[correct, MR, distanced]"
302,mrobbins,cube2,71712.0,51080.0,77344.0,"[correct, MR, distanced]"
278,mrobbins,cube2,71808.0,51488.0,76504.0,"[correct, MR]"
